In [2]:
import datasets

example_dataset = datasets.load_dataset('parquet', data_files='/mnt/data/share/data1/gui-r1/androidcontrol_high_test.parquet')['train']
print(example_dataset[0].keys())
print('history', example_dataset[0]['history'])
print('gt_action', example_dataset[0]['gt_action'])
print(example_dataset[0]['gt_bbox'])
print(example_dataset[0]['group'])
print(example_dataset[0]['ui_type'])



dict_keys(['image', 'history', 'instruction', 'gt_action', 'gt_bbox', 'gt_input_text', 'group', 'ui_type'])
history  
Step 1: Click on the three bar menu at the top left corner of the screen


gt_action click
[205, 652]
android
click


# Android Control

In [1]:
import json
import os

with open('/mnt/data/share/data1/android_control/android_control/android_control.json', 'r') as f:
    raw_data = json.load(f)
new_dataset = []

action_transform = {
    'click': 'click',
    'long_press': 'long_press',
    'scroll': 'swipe',
    'input_text': 'type',
    'open_app': 'open',
    'navigate_back': 'system_button',
    'navigate_home': 'system_button',
}

def make_history(action, bbox, input_txt):
    if action in ['click', 'long_press']:
        step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"coordinate\": \"{bbox}\"}}}}'
    elif action in ['type', 'open']:
        step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"text\": \"{input_txt}\"}}}}'
    elif action in ['system_button']:
        step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"button\": \"{input_txt}\"}}}}'
    elif action in ['wait']:
        step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"time\": \"3\"}}}}'
    elif action in ['swipe']:
        if input_txt == 'up':
            step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"coordinate\": [546, 1204], \"coordinate2\": [546, 604]}}}}'
        elif input_txt == 'down':
            step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"coordinate\": [546, 604], \"coordinate2\": [546, 1204]}}}}'
        elif input_txt == 'left':
            step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"coordinate\": [546, 1204], \"coordinate2\": [0, 1204]}}}}'
        elif input_txt == 'right':
            step_info = f'{{\"name\": \"mobile_use\", \"arguments\": {{\"action\": \"{action}\", \"coordinate\": [0, 1204], \"coordinate2\": [546, 1204]}}}}'
    else:
        print('Unknown action type:', action)
        return ""
    return step_info


for episode in raw_data:
    history = ""
    goal = episode['goal']
    for step in range(len(episode['images'])):
        img_file = episode['images'][step]
        img_file = os.path.join('/mnt/data/share/data1/android_control/android_control/images', img_file)
        with open(img_file, 'rb') as f:
            img_bytes = f.read()
        
        gt_action = episode['actions'][step]
        gt_action = json.loads(gt_action)
        if gt_action['action_type'] in ['click', 'long_press']:
            action = gt_action['action_type']
            bbox = [gt_action['x'], gt_action['y']]
            input_txt = ""
        elif gt_action['action_type'] in ['input_text', 'scroll', 'open_app']:
            action = action_transform[gt_action['action_type']]
            bbox = [-1,-1]
            # input_txt = gt_action[]
            # get the value other than key action_type
            input_txt = list(gt_action.values())[1]
        elif gt_action['action_type'] in ['wait']:
            action = gt_action['action_type']
            bbox = [-1,-1]
            input_txt = ""
        elif gt_action['action_type'] in ['navigate_back', 'navigate_home']:
            action = action_transform[gt_action['action_type']]
            bbox = [-1,-1]
            input_txt = gt_action['action_type'].split('_')[-1]
            input_txt = input_txt.capitalize()
        else:
            print('Unknown action type:', gt_action['action_type'])

        history += (f"Step {step+1}: " + make_history(action, bbox, input_txt) + "; ")

        new_dataset.append({
            'image': {'bytes': img_bytes},
            'instruction': goal,
            'history': history if len(history) > 0 else "None",
            'gt_action': action,
            'gt_bbox': bbox,
            'gt_input_text': input_txt,
            'ui_type': 'android_control',
            'group': 'high'
        })

In [2]:
print('Total steps:', len(new_dataset))

Total steps: 83848


In [3]:
print(new_dataset[13]['history'])

Step 1: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 1204], "coordinate2": [0, 1204]}}; Step 2: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 604], "coordinate2": [546, 1204]}}; Step 3: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 604], "coordinate2": [546, 1204]}}; Step 4: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 604], "coordinate2": [546, 1204]}}; Step 5: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 604], "coordinate2": [546, 1204]}}; Step 6: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 604], "coordinate2": [546, 1204]}}; Step 7: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 604], "coordinate2": [546, 1204]}}; Step 8: {"name": "mobile_use", "arguments": {"action": "swipe", "coordinate": [546, 604], "coordinate2": [546, 1204]}}; Step 9: {"name": "mobile_use", "arguments

In [ ]:
import datasets

new_dataset = datasets.Dataset.from_list(new_dataset)
new_dataset.to_parquet('/mnt/data/share/data1/gui-r1/android_control_high4filter.parquet')

/mnt/data/home/zoulexiao/anaconda3/envs/easyr1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# AMEX

In [ ]:
import json
import os

origin_json_dir = '/mnt/data/share/data1/AMEX/AMEX/instruction_anno'

new_dataset = []
action_transform = {
    'TAP': 'click',
    'SCROLL': 'swipe',
    'TYPE': 'type',
    'PRESS_BACK': 'system_button',
    'PRESS_HOME': 'system_button',
    'PRESS_ENTER': 'system_button',
    'TASK_COMPLETE': 'terminate',
    'TASK_IMPOSSIBLE': 'terminate'
}


for episode_file in os.listdir(origin_json_dir):
    if not episode_file.endswith('.json'):
        continue
    episode_id = episode_file.split('.')[0]
    with open(os.path.join(origin_json_dir, episode_file), 'r') as f:
        episode_data = json.load(f)
        # print(episode_data['instruction'])
    for step in episode_data['steps']:
        step_id = step['step_id']
        
        if step['action'] == 'TAP':
            point = step['touch_coord']
            element_file = step['image_path'].replace('.png', '.json')
            # print('step_id', step_id, 'point', point, 'element_file', element_file)
            with open(os.path.join('/mnt/data/share/data1/AMEX/AMEX/element_anno', element_file), 'r') as ef:
                element_data = json.load(ef)
                element_list = element_data['clickable_elements']
                gt_bbox = []
                for element in element_list:
                    bbox = element['bbox']
                    if bbox[0] <= point[0] <= bbox[2] and bbox[1] <= point[1] <= bbox[3]:
                        gt_bbox.append(bbox)
            
            new_dataset.append({
                'episode_id': episode_id,
                'step_id': step_id,
                'gt_bbox': gt_bbox
            })     

In [10]:
print(len(new_dataset))
print(len([item for item in new_dataset if len(item['gt_bbox']) != 1]))

24815
2683
